# 04 -- Altair

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- construire des graphiques avec la grammaire declarative d'Altair ;
- mapper des donnees Pandas sur des encodings visuels ;
- creer des graphiques a barres, lignes, scatter, aires ;
- combiner des graphiques (layer, hconcat, vconcat, facet) ;
- ajouter de l'interactivite (selection, zoom, tooltips) ;
- sauvegarder en HTML, PNG, SVG ;
- utiliser les themes et le theming.

## Prerequis -- ce que vous devez maitriser

- Pandas (DataFrame, Series) ;
- les bases statistiques (moyenne, comptage).

Ce que ce notebook **ne couvrira pas** :

- Matplotlib (voir `01_Matplotlib_OO_complet.ipynb`) ;
- Plotly (voir `03_Plotly.ipynb`) ;
- les visualisations 3D.

## Plan

1. Installation et philosophie
2. Premier graphique
3. Encodings
4. Types de marques
5. Barres et groupes
6. Lignes temporelles
7. Scatter et bubbles
8. Agregations
9. Layer et combinaisons
10. Facets
11. Interactivite
12. Themes
13. Sauvegarde
14. Synthese
15. Exercices

---

## 1. Installation et philosophie

In [ ]:
# pip install altair vega_datasets
import altair as alt
alt.__version__

Altair est une bibliotheque **declarative** : on decrit *quoi* afficher (mapping donnees -> encodings visuels), pas *comment* dessiner.

Elle s'appuie sur Vega-Lite en arriere-plan.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'categorie': ['A', 'B', 'C', 'D'],
    'valeur':    [4, 7, 2, 9],
})
df

---

## 2. Premier graphique

In [ ]:
alt.Chart(df).mark_bar().encode(
    x='categorie',
    y='valeur',
)

Un graphique Altair est compose de :

1. les **donnees** (un `DataFrame`) ;
2. un **type de marque** (`mark_bar`, `mark_line`, ...) ;
3. des **encodings** (quelle colonne mappee sur quel axe/couleur/taille).

---

## 3. Encodings

In [ ]:
alt.Chart(df).mark_bar().encode(
    x='categorie:N',   # N = nominal
    y='valeur:Q',      # Q = quantitatif
    color='categorie:N',
)

### Types d'encodings

| Suffixe | Type | Exemple |
|---|---|---|
| `:N` | nominal (categoriel non ordonne) | couleurs, villes |
| `:O` | ordinal (categoriel ordonne) | petit/moyen/grand |
| `:Q` | quantitatif | prix, taille |
| `:T` | temporel | dates, timestamps |

Altair detecte souvent automatiquement, mais on peut forcer.

---

## 4. Types de marques

In [ ]:
alt.Chart(df).mark_point().encode(x='categorie', y='valeur')

In [ ]:
alt.Chart(df).mark_line().encode(x='categorie', y='valeur')

In [ ]:
alt.Chart(df).mark_area().encode(x='categorie', y='valeur')

In [ ]:
alt.Chart(df).mark_circle(size=200).encode(x='categorie', y='valeur')

In [ ]:
alt.Chart(df).mark_text().encode(x='categorie', y='valeur', text='valeur')

---

## 5. Barres et groupes

In [ ]:
ventes = pd.DataFrame({
    'mois':    ['Jan', 'Fev', 'Mar'] * 2,
    'produit': ['A', 'A', 'A', 'B', 'B', 'B'],
    'ca':      [100, 120, 150, 80, 95, 110],
})
ventes

In [ ]:
# Barres groupees
alt.Chart(ventes).mark_bar().encode(
    x='mois:N',
    y='ca:Q',
    color='produit:N',
    xOffset='produit:N',  # force des groupes cote a cote
)

In [ ]:
# Barres empilees (par defaut avec color)
alt.Chart(ventes).mark_bar().encode(
    x='mois:N',
    y='ca:Q',
    color='produit:N',
)

---

## 6. Lignes temporelles

In [ ]:
import numpy as np
rng = pd.date_range('2024-01-01', periods=30)
ts = pd.DataFrame({'date': rng, 'valeur': np.cumsum(np.random.default_rng(0).standard_normal(30))})
ts.head()

In [ ]:
alt.Chart(ts).mark_line().encode(
    x='date:T',
    y='valeur:Q',
)

In [ ]:
# Points + ligne
alt.Chart(ts).mark_line(point=True).encode(
    x='date:T',
    y='valeur:Q',
)

---

## 7. Scatter et bubbles

In [ ]:
from vega_datasets import data
cars = data.cars()
cars[['Name', 'Horsepower', 'Miles_per_Gallon', 'Origin']].head()

In [ ]:
alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon'],
)

In [ ]:
# Bubble chart
alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    size='Weight_in_lbs:Q',
    color='Origin:N',
)

---

## 8. Agregations

In [ ]:
# Moyenne par origine
alt.Chart(cars).mark_bar().encode(
    x='Origin:N',
    y='mean(Horsepower):Q',
)

In [ ]:
# Comptage
alt.Chart(cars).mark_bar().encode(
    x='Origin:N',
    y='count():Q',
)

In [ ]:
# Histogramme
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Horsepower:Q', bin=alt.Bin(maxbins=20)),
    y='count():Q',
)

---

## 9. Layer et combinaisons

In [ ]:
bars = alt.Chart(df).mark_bar().encode(x='categorie', y='valeur')
text = alt.Chart(df).mark_text(dy=-10).encode(x='categorie', y='valeur', text='valeur')
bars + text  # layer

In [ ]:
# Concatenation horizontale
chart_a = alt.Chart(df).mark_bar().encode(x='categorie', y='valeur')
chart_b = alt.Chart(df).mark_point().encode(x='categorie', y='valeur')
chart_a | chart_b

In [ ]:
# Concatenation verticale
chart_a & chart_b

---

## 10. Facets

In [ ]:
alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
).facet(column='Origin:N')

In [ ]:
# Facet avec grille
alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
).facet(
    column='Origin:N',
    row='Cylinders:O',
)

---

## 11. Interactivite

In [ ]:
# Zoom et pan
alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
).interactive()

### Selection avec highlight

In [ ]:
brush = alt.selection_interval()

alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(brush, 'Origin:N', alt.value('lightgray')),
).add_params(brush)

---

## 12. Themes

In [ ]:
# alt.themes.enable('dark')
# alt.themes.enable('fivethirtyeight')
list(alt.themes.names())

In [ ]:
alt.themes.enable('default')
alt.Chart(df).mark_bar().encode(x='categorie', y='valeur')

---

## 13. Sauvegarde

In [ ]:
chart = alt.Chart(df).mark_bar().encode(x='categorie', y='valeur')
import tempfile, pathlib
out = pathlib.Path(tempfile.gettempdir()) / 'chart.html'
chart.save(str(out))
print('Sauvegarde :', out)

### Formats supportes

- `.html` (natif, interactif)
- `.json` (spec Vega-Lite)
- `.png` / `.svg` (necessite `vl-convert-python`)

In [ ]:
# chart.save('chart.png')  # pip install vl-convert-python
print('Pour PNG/SVG : pip install vl-convert-python')

---

## Synthese

| Element | Altair |
|---|---|
| Objet principal | `alt.Chart(df)` |
| Type de marque | `.mark_bar()`, `.mark_line()`, ... |
| Encodings | `.encode(x=..., y=..., color=...)` |
| Type explicite | `'colonne:N/O/Q/T'` |
| Agregation | `'mean(col):Q'`, `'count():Q'` |
| Layer | `chart_a + chart_b` |
| Cote a cote | `chart_a | chart_b` |
| Empile | `chart_a & chart_b` |
| Facets | `.facet(column=..., row=...)` |
| Interactif | `.interactive()` |

### Regles a retenir

- **Declaratif** : on decrit ce qu'on veut, Vega-Lite genere le SVG/Canvas.
- **Toujours passer par un DataFrame**.
- **Types explicites** (`:N`, `:Q`...) evitent les surprises.
- `+`, `|`, `&` pour combiner ; `.facet` pour grille de sous-graphes.

---

## Exercices

### Exercice 1 (facile) -- Bar chart

Creez un DataFrame avec 5 villes et leur population, affichez un graphique a barres trie par population decroissante.

In [ ]:
import altair as alt
import pandas as pd

# TODO

<details>
<summary>Voir la correction</summary>

```python
import altair as alt
import pandas as pd

df = pd.DataFrame({
    'ville': ['Paris', 'Marseille', 'Lyon', 'Toulouse', 'Nice'],
    'population': [2148000, 870000, 520000, 490000, 340000],
})
alt.Chart(df).mark_bar().encode(
    x=alt.X('ville:N', sort='-y'),
    y='population:Q',
)
```

</details>

### Exercice 2 (facile) -- Scatter avec couleur

A partir du dataset `vega_datasets.data.cars()`, faites un scatter `Horsepower` vs `Acceleration`, colorie par `Origin`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from vega_datasets import data
import altair as alt

cars = data.cars()
alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Acceleration:Q',
    color='Origin:N',
    tooltip=['Name'],
)
```

</details>

### Exercice 3 (moyen) -- Histogramme

Sur le dataset `cars`, creez un histogramme du `Miles_per_Gallon` avec 20 bins, colorie par `Origin`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from vega_datasets import data
import altair as alt

cars = data.cars()
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon:Q', bin=alt.Bin(maxbins=20)),
    y='count():Q',
    color='Origin:N',
)
```

</details>

### Exercice 4 (moyen) -- Layer avec moyenne

Sur `cars`, superposez un scatter plot et une ligne horizontale representant la moyenne de `Horsepower`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from vega_datasets import data
import altair as alt

cars = data.cars()
points = alt.Chart(cars).mark_circle().encode(
    x='Miles_per_Gallon:Q',
    y='Horsepower:Q',
)
mean_line = alt.Chart(cars).mark_rule(color='red').encode(
    y='mean(Horsepower):Q',
)
points + mean_line
```

</details>

### Exercice 5 (difficile) -- Dashboard interactif

Construisez un dashboard Altair combinant :
1. un scatter Horsepower/MPG avec une selection intervalle ;
2. un histogramme de MPG qui met en valeur la selection.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from vega_datasets import data
import altair as alt

cars = data.cars()
brush = alt.selection_interval()

scatter = alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(brush, 'Origin:N', alt.value('lightgray')),
).add_params(brush)

hist = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon:Q', bin=alt.Bin(maxbins=20)),
    y='count():Q',
).transform_filter(brush)

scatter & hist
```

</details>

---

## Ressources externes

- [Documentation Altair](https://altair-viz.github.io/)
- [Gallery](https://altair-viz.github.io/gallery/index.html)
- [Vega-Lite](https://vega.github.io/vega-lite/)
- [vega_datasets](https://github.com/altair-viz/vega_datasets)